# Teledyne LeCroy T3PS3000 DC power supply

The T3PS3000 has two programmable outputs (CH1 and CH2, each 0-32 V and 0-3.2 A) and a third output whose voltage is selected on the front panel (2.5, 3.3, or 5 V). The QCoDeS driver exposes these as `ch1`, `ch2`, and `ch3`. The `channels` tuple contains only CH1 and CH2.

You need a connected instrument and a working VISA backend. Replace the example VISA address with your device's USB or LAN address. Check wiring and choose appropriate voltage and current limits before enabling any output. Initialization does not reset the supply or change output settings, and closing the connection does not switch outputs off.

The output-control cells enable physical outputs when run. Check the connected circuit before running them or using Run All. Timer enable commands remain commented out.

Close the NI VISA test-panel session before connecting. The driver waits 0.1 seconds after each write command before returning, so subsequent writes or queries do not arrive too soon. Queries separately wait one second between writing the query and reading its reply, including the initial `*IDN?`; reading immediately can hang USB communication on this supply. Restart the kernel after updating the driver so the new timing is loaded.

This Windows setup selects NI-VISA directly with `visalib` because the default shared VISA dispatcher did not reliably return identification. On other systems, replace this path with the installed VISA library or omit `visalib` to use the default backend.

The connection below skips VISA device-clear (`device_clear=False`) as a USB troubleshooting precaution; this does not disable outputs or reset the supply. The three-second timeout requests a VISA I/O limit, but a blocked native USB operation may not honor it. Progress messages distinguish instrument construction from station registration. Station registration uses cached values rather than querying every parameter.

In [1]:
from qcodes_contrib_drivers.drivers.Teledyne import TeledyneT3PS3000
from qcodes.station import Station


In [ ]:
resource_name = "USB0::0xF4EC::0x1430::T0103C19150168::INSTR"
print("Opening VISA resource and querying *IDN?...", flush=True)
pwr = TeledyneT3PS3000("pwr", resource_name, device_clear=False, timeout=3)
print("Instrument initialized; registering with station...", flush=True)
station = Station()
station.add_component(pwr, update_snapshot=False)
print("Station ready.", flush=True)

Opening VISA resource and querying *IDN?...
Connected to: T3PS3000 T0103C19150168 (serial:1.01.01.02.05R1, firmware:V3.0) in 1.12s
Instrument initialized; registering with station...
Station ready.


## Independent operation

Disable timers and outputs before configuring. Channel parameters always address the named channel, regardless of `selected_channel`. Valid tracking modes are `"independent"`, `"series"`, and `"parallel"`. This example uses independent mode. In tracking modes the setpoints remain per-channel values; do not program a combined 64 V or 6.4 A value into a single channel.

In [4]:
for channel in pwr.channels:
    channel.timer_enabled(False)
    channel.output(False)
pwr.ch3.output(False)
pwr.operation_mode("independent")

pwr.ch1.voltage(5.0)
pwr.ch1.current(1.0)
pwr.ch2.voltage(12.0)
pwr.ch2.current(0.5)

for channel in pwr.channels:
    print(channel.name, channel.voltage(), "V", channel.current(), "A")

pwr_ch1 5.0 V 1.0 A
pwr_ch2 12.0 V 0.5 A


## Status and measurements

Read output state, regulation mode, and actual voltage/current/power through channel parameters. CH3 output control is **write-only**: the instrument does not report its output state in the status register, so the driver does not invent a readback value. CH3 voltage is selected on the front panel.

Run an output-enable cell only after checking the connected circuit.

In [ ]:
print(f"Status: {pwr.status():#06x}")
print("Operation mode:", pwr.operation_mode())
print("Selected channel:", pwr.selected_channel())

for channel in pwr.channels:
    print(channel.name, "enabled:", channel.output(), "mode:", channel.mode())
    print(channel.measured_voltage(), "V", channel.measured_current(), "A")
    print(channel.measured_power(), "W")

Status: 0x0004
Operation mode: independent
Selected channel: CH1
pwr_ch1 enabled: False mode: CV
0.0 V 0.0 A
0.0 W
pwr_ch2 enabled: False mode: CV
0.0 V 0.0 A
0.0 W


## Output control
Turn channels 1 & 2 on

In [3]:
pwr.ch1.output(True)
pwr.ch2.output(True)

Turn channels 1 & 2 off

In [4]:
pwr.ch1.output(False)
pwr.ch2.output(False)

### Channel 3

Check the front-panel voltage selection (2.5, 3.3, or 5 V) and the connected circuit before enabling CH3. Its output state cannot be queried remotely.

Turn channel 3 on.

In [5]:
pwr.ch3.output(True)

Turn channel 3 off.

In [6]:
pwr.ch3.output(False)

## Timer sequences

Each adjustable channel has five timer steps. A step specifies voltage, current limit, and duration (0-10000 seconds). All values are validated before transmission. Timers can only be enabled in independent mode.

Program all five steps so an old configuration cannot contribute unexpected later steps. The unused final steps are set to zero voltage, current, and duration. `set_timer` only programs a step; it does not start the timer. `get_timer` returns `(voltage, current, duration)` as floats.

In [ ]:
steps = [
    (5.0, 0.5, 10.0),
    (10.0, 1.0, 20.0),
    (3.3, 0.2, 15.0),
    (0.0, 0.0, 0.0),
    (0.0, 0.0, 0.0),
]
for group, (voltage, current, duration) in enumerate(steps, start=1):
    pwr.ch1.set_timer(group, voltage, current, duration)

print("First timer step:", pwr.ch1.get_timer(1))
# pwr.ch1.timer_enabled(True)

## Network configuration

Query LAN configuration using ordinary parameters. Static address setters validate IPv4 syntax and require DHCP to be disabled. Changing DHCP or an IP address may disconnect a LAN session; perform changes over USB when possible and reconnect at the new address. The example only reads these values.

In [ ]:
print("DHCP:", pwr.dhcp())
print("IP:", pwr.ip_address())
print("Subnet mask:", pwr.subnet_mask())
print("Gateway:", pwr.gateway())

# pwr.dhcp(False)
# pwr.subnet_mask("255.255.255.0")
# pwr.gateway("192.0.2.1")
# pwr.ip_address("192.0.2.10")

snapshot = station.snapshot(update="All")

## Stop timers, disable outputs, and close

Run cleanup even if an earlier cell raises an error. If communication fails during shutdown, check the outputs on the front panel; closing VISA alone is not an output-off command.

In [7]:
try:
    for channel in pwr.channels:
        channel.timer_enabled(False)
        channel.output(False)
    pwr.ch3.output(False)
finally:
    pwr.close()